<a href="https://colab.research.google.com/github/RavisriWelabada/ML-PROJECT---CHARITY-DONOR-ANALYSIS-/blob/main/ML_CHARITY_DONOR_ANALYSIS_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHARITY DONOR ANALYSIS

---


  Problem: Identify the most beneficial individuals to contact

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist, cosine
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import chi2_contingency

from sklearn.preprocessing import (LabelEncoder, StandardScaler,
                                   MinMaxScaler, PolynomialFeatures)
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, KFold, StratifiedKFold)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              f1_score, mean_squared_error, r2_score,
                              silhouette_score, mean_absolute_error)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import (LogisticRegression, LinearRegression,
                                   Ridge, Lasso, BayesianRidge)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.svm import SVC, SVR
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               AdaBoostClassifier, GradientBoostingClassifier,
                               GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.utils import resample
import itertools
import os

print("=" * 70)
print("  CHARITY DONOR TARGETING — COMPREHENSIVE ANALYTICAL PIPELINE")
print("=" * 70)



1. LOAD DATA

In [ ]:
print("\n[STEP 1] Loading data...")
df = pd.read_csv("https://raw.githubusercontent.com/RavisriWelabada/ML-PROJECT---CHARITY-DONOR-ANALYSIS-/main/donor_data.csv")
print(f"  Dataset shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")

2. DESCRIPTIVE / SUMMARY STATISTICS

In [ ]:
print("\n[STEP 2] Summary Statistics")
print("-" * 50)
print(df.describe(include="all").T.to_string())

# Target class balance
donor_count   = df["TARGET_B"].sum()
non_donor_cnt = len(df) - donor_count
print(f"\n  Donors: {donor_count} ({donor_count/len(df)*100:.1f}%)")
print(f"  Non-donors: {non_donor_cnt} ({non_donor_cnt/len(df)*100:.1f}%)")

3. CATEGORICAL & NUMERICAL SPLIT

In [ ]:
cat_cols = ["URBANICITY", "SES", "CLUSTER_CODE", "HOME_OWNER",
            "DONOR_GENDER", "OVERLAY_SOURCE", "RECENCY_STATUS_96NK"]
num_cols = [c for c in df.columns
            if c not in cat_cols + ["TARGET_B", "TARGET_D", "CONTROL_NUMBER"]]

print(f"\n  Numeric cols ({len(num_cols)}): {num_cols[:8]} ...")
print(f"  Categorical cols ({len(cat_cols)}): {cat_cols}")

4. MISSING VALUE ANALYSIS & IMPUTATION

In [ ]:
print("\n[STEP 3] Missing Value Analysis & Imputation")
print("-" * 50)
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df = pd.DataFrame({"Missing": miss, "Pct": miss_pct}).query("Missing > 0")
print(miss_df)

# 4a. Predict missing DONOR_AGE using KNN regression
print("\n  [4a] Predicting DONOR_AGE with KNN Regressor (predictive imputation)...")
age_features = ["MONTHS_SINCE_FIRST_GIFT", "LIFETIME_GIFT_COUNT",
                "INCOME_GROUP", "MONTHS_SINCE_ORIGIN"]
# Temporarily fill INCOME_GROUP for imputation model
df["INCOME_GROUP"].fillna(df["INCOME_GROUP"].median(), inplace=True)

age_known   = df[df["DONOR_AGE"].notna()][age_features + ["DONOR_AGE"]].copy()
age_unknown = df[df["DONOR_AGE"].isna()][age_features].copy()

knn_age = KNeighborsRegressor(n_neighbors=5)
knn_age.fit(age_known[age_features], age_known["DONOR_AGE"])
df.loc[df["DONOR_AGE"].isna(), "DONOR_AGE"] = knn_age.predict(age_unknown)
print(f"    DONOR_AGE missing → predicted via KNN. Remaining NaN: {df['DONOR_AGE'].isna().sum()}")

# 4b. Fill remaining numeric with median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        df[col].fillna(med, inplace=True)
        print(f"    {col}: filled {df[col].isna().sum()} NaN → median ({med:.2f})")

# 4c. Fill WEALTH_RATING with median
df["WEALTH_RATING"].fillna(df["WEALTH_RATING"].median(), inplace=True)

# 4d. Fill categorical with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0 or (df[col] == "?").sum() > 0:
        df[col] = df[col].replace("?", np.nan)
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"    {col}: filled with mode ({mode_val})")

# Fix MONTHS_SINCE_LAST_PROM_RESP
df["MONTHS_SINCE_LAST_PROM_RESP"].fillna(df["MONTHS_SINCE_LAST_PROM_RESP"].median(), inplace=True)
print(f"\n  Total missing after imputation: {df.isnull().sum().sum()}")

5. OUTLIER HANDLING — IQR METHOD

In [ ]:
print("\n[STEP 4] Outlier Handling — IQR Method")
print("-" * 50)
iqr_cols = ["DONOR_AGE", "LIFETIME_GIFT_AMOUNT", "LAST_GIFT_AMT",
            "RECENT_AVG_GIFT_AMT", "LIFETIME_AVG_GIFT_AMT"]
outlier_report = {}
for col in iqr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = n_out
    # Cap (Winsorise) instead of drop
    df[col] = df[col].clip(lower, upper)
    print(f"  {col}: {n_out} outliers capped to [{lower:.2f}, {upper:.2f}]")

6. EXPLORATORY PLOTS

In [ ]:
print("\n[STEP 5] Generating EDA plots...")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Donor Data — Exploratory Analysis", fontsize=16, fontweight="bold")

# Distribution of age by donor status
donors_df = df[df["TARGET_B"] == 1]
non_donors = df[df["TARGET_B"] == 0]
axes[0,0].hist(donors_df["DONOR_AGE"], bins=30, alpha=0.6, label="Donor", color="steelblue")
axes[0,0].hist(non_donors["DONOR_AGE"], bins=30, alpha=0.6, label="Non-Donor", color="salmon")
axes[0,0].set_title("Age Distribution by Donor Status")
axes[0,0].legend()
axes[0,0].set_xlabel("Age")

# Gift amount distribution (donors only)
axes[0,1].hist(df[df["TARGET_D"].notna()]["TARGET_D"], bins=40, color="green", alpha=0.7)
axes[0,1].set_title("Donation Amount Distribution (Donors)")
axes[0,1].set_xlabel("$ Donated")
axes[0,1].axvline(df["TARGET_D"].mean(), color="red", linestyle="--", label=f"Mean={df['TARGET_D'].mean():.1f}")
axes[0,1].legend()

# Correlation heatmap
corr_cols = ["DONOR_AGE", "LIFETIME_GIFT_AMOUNT", "LIFETIME_GIFT_COUNT",
             "RECENT_RESPONSE_PROP", "LAST_GIFT_AMT", "TARGET_B"]
corr_mat = df[corr_cols].corr()
sns.heatmap(corr_mat, ax=axes[0,2], annot=True, fmt=".2f", cmap="coolwarm", center=0)
axes[0,2].set_title("Correlation Heatmap (Key Features)")

# Class imbalance
labels = ["Non-Donor (75%)", "Donor (25%)"]
sizes  = [non_donor_cnt, donor_count]
axes[1,0].pie(sizes, labels=labels, autopct="%1.1f%%",
              colors=["#ff9999","#66b3ff"], startangle=90)
axes[1,0].set_title("Class Balance (TARGET_B)")

# Donation by gender
gender_map = df.groupby("DONOR_GENDER")["TARGET_B"].mean() * 100
gender_map.plot(kind="bar", ax=axes[1,1], color="teal", edgecolor="black")
axes[1,1].set_title("Donation Rate by Gender (%)")
axes[1,1].set_xlabel("Gender")
axes[1,1].set_ylabel("% Donated")

# Lifetime gift by wealth rating
wealth_gift = df.groupby("WEALTH_RATING")["LIFETIME_GIFT_AMOUNT"].mean()
wealth_gift.plot(kind="bar", ax=axes[1,2], color="purple", edgecolor="black")
axes[1,2].set_title("Avg Lifetime Gift by Wealth Rating")
axes[1,2].set_xlabel("Wealth Rating")

# Box plot — Last gift by donor status
df.boxplot(column="LAST_GIFT_AMT", by="TARGET_B", ax=axes[2,0])
axes[2,0].set_title("Last Gift Amount by Donor Status")
axes[2,0].set_xlabel("Donor (1) / Non-Donor (0)")

# Donation rate by urbanicity
urb_rate = df.groupby("URBANICITY")["TARGET_B"].mean() * 100
urb_rate.sort_values().plot(kind="barh", ax=axes[2,1], color="darkorange")
axes[2,1].set_title("Donation Rate by Urbanicity (%)")

# Recency status vs donation
recency_rate = df.groupby("RECENCY_STATUS_96NK")["TARGET_B"].mean() * 100
recency_rate.sort_values().plot(kind="bar", ax=axes[2,2], color="crimson")
axes[2,2].set_title("Donation Rate by Recency Status (%)")
axes[2,2].set_xlabel("Recency Status")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_EDA.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 01_EDA.png")

7. FEATURE ENGINEERING

In [ ]:
print("\n[STEP 6] Feature Engineering")
print("-" * 50)
df["ENGAGEMENT_SCORE"] = (df["RECENT_RESPONSE_PROP"] * 0.4 +
                          df["LIFETIME_GIFT_COUNT"]    * 0.3 +
                          df["RECENT_STAR_STATUS"]     * 0.3)
df["GIFT_VELOCITY"]    = df["LIFETIME_GIFT_AMOUNT"] / (df["MONTHS_SINCE_FIRST_GIFT"] + 1)
df["RECENCY_FREQ_SCORE"] = df["RECENT_RESPONSE_COUNT"] * df["RECENT_RESPONSE_PROP"]
df["WEALTH_INCOME_RATIO"] = df["WEALTH_RATING"] / (df["INCOME_GROUP"] + 1)
df["PROM_RESPONSE_EFFICIENCY"] = (df["LIFETIME_GIFT_COUNT"] /
                                  (df["LIFETIME_PROM"] + 1))
df["IS_HIGH_VALUE"]    = (df["LIFETIME_GIFT_AMOUNT"] > df["LIFETIME_GIFT_AMOUNT"].quantile(0.75)).astype(int)
df["RECENT_CARD_FLAG"] = (df["RECENT_CARD_RESPONSE_COUNT"] > 0).astype(int)
df["AGE_GROUP"]        = pd.cut(df["DONOR_AGE"], bins=[0,45,60,75,120],
                                labels=["Young","Middle","Senior","Elderly"])

new_features = ["ENGAGEMENT_SCORE","GIFT_VELOCITY","RECENCY_FREQ_SCORE",
                "WEALTH_INCOME_RATIO","PROM_RESPONSE_EFFICIENCY",
                "IS_HIGH_VALUE","RECENT_CARD_FLAG"]
print(f"  Created {len(new_features)} new features: {new_features}")

8. ENCODING CATEGORICAL FEATURES

In [ ]:
print("\n[STEP 7] Encoding Categorical Features")
print("-" * 50)
le = LabelEncoder()
encode_cols = cat_cols + ["AGE_GROUP"]
for col in encode_cols:
    df[col] = df[col].astype(str)
    df[col + "_ENC"] = le.fit_transform(df[col])
    print(f"  {col} → {col}_ENC  ({df[col].nunique()} categories)")

9. FEATURE SELECTION

In [ ]:
print("\n[STEP 8] Feature Selection")
print("-" * 50)
feature_cols = ([c for c in num_cols if c in df.columns] +
                new_features +
                [c + "_ENC" for c in encode_cols])
feature_cols = [c for c in feature_cols if c in df.columns]

X_all = df[feature_cols].fillna(0)
y_class = df["TARGET_B"]

# Chi-squared (categorical target)
print("  SelectKBest — f_classif (ANOVA F-test):")
selector = SelectKBest(f_classif, k=20)
selector.fit(X_all, y_class)
scores_df = pd.DataFrame({"Feature": feature_cols,
                          "Score": selector.scores_}).sort_values("Score", ascending=False)
top_features = scores_df.head(20)["Feature"].tolist()
print(scores_df.head(10).to_string(index=False))

# Mutual Information
print("\n  Mutual Information Scores (top 10):")
mi_scores = mutual_info_classif(X_all, y_class, random_state=42)
mi_df = pd.DataFrame({"Feature": feature_cols, "MI_Score": mi_scores}).sort_values("MI_Score", ascending=False)
print(mi_df.head(10).to_string(index=False))

# Combine top features
top_features_final = list(set(scores_df.head(15)["Feature"].tolist() +
                               mi_df.head(15)["Feature"].tolist()))
print(f"\n  Final selected features: {len(top_features_final)}")

# Chi-sq test for categorical features vs TARGET_B
print("\n  Chi-Squared Tests (Categorical Features):")
for col in cat_cols:
    ct = pd.crosstab(df[col], df["TARGET_B"])
    chi2_stat, p_val, _, _ = chi2_contingency(ct)
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else ""))
    print(f"    {col:25s}: χ²={chi2_stat:8.2f}, p={p_val:.4f} {sig}")



10. BOOTSTRAPPING

In [ ]:
print("\n[STEP 9] Bootstrapping — Mean Estimation (1000 samples)")
print("-" * 50)
np.random.seed(42)
boot_means = [np.mean(resample(df["LAST_GIFT_AMT"], random_state=i)) for i in range(1000)]
print(f"  Bootstrap Mean LAST_GIFT_AMT: {np.mean(boot_means):.2f}")
print(f"  95% CI: [{np.percentile(boot_means, 2.5):.2f}, {np.percentile(boot_means, 97.5):.2f}]")



11. MANUAL SMOTE (class balancing)

In [ ]:

print("\n[STEP 10] Class Balancing — Manual SMOTE-style Oversampling")
print("-" * 50)
X = df[top_features_final].fillna(0)
y = df["TARGET_B"]

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_maj = X_train_raw[y_train_raw == 0]
X_min = X_train_raw[y_train_raw == 1]
n_oversample = len(X_maj) - len(X_min)

# Interpolation-based synthetic samples
np.random.seed(42)
idx = np.random.choice(len(X_min), n_oversample, replace=True)
noise = np.random.normal(0, 0.01, (n_oversample, X_min.shape[1]))
X_synthetic = X_min.values[idx] + noise
X_bal = pd.DataFrame(np.vstack([X_train_raw.values,
                                  X_synthetic]), columns=X_train_raw.columns)
y_bal = pd.Series(np.hstack([y_train_raw.values,
                               np.ones(n_oversample, dtype=int)]))

print(f"  Before: {y_train_raw.value_counts().to_dict()}")
print(f"  After:  {y_bal.value_counts().to_dict()}")

12. SCALING

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_bal)
X_test_sc  = scaler.transform(X_test)
print(f"\n  StandardScaler applied. Train shape: {X_train_sc.shape}")


13. PCA

In [ ]:
print("\n[STEP 11] PCA — Dimensionality Reduction")
print("-" * 50)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)
print(f"  Original features: {X_train_sc.shape[1]}")
print(f"  PCA components (95% variance): {X_train_pca.shape[1]}")
print(f"  Variance explained per component: {np.round(pca.explained_variance_ratio_[:5]*100,1)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_*100)
ax1.set_title("PCA — Explained Variance per Component")
ax1.set_xlabel("Component"); ax1.set_ylabel("Variance (%)")
ax2.plot(np.cumsum(pca.explained_variance_ratio_)*100, marker="o")
ax2.axhline(95, color="red", linestyle="--", label="95% threshold")
ax2.set_title("Cumulative Explained Variance"); ax2.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_PCA.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 02_PCA.png")

14. CLUSTERING

In [ ]:
print("\n[STEP 12] Clustering Analysis")
print("-" * 50)

# Use PCA-reduced data for clustering
X_cluster = X_train_pca[:, :5]   # first 5 components

# 14a. Cosine Distance / Proximity Matrix (sample)
print("  [12a] Cosine Distance (Proximity Matrix — first 6 samples):")
sample = X_cluster[:6]
cos_dist = cdist(sample, sample, metric="cosine")
print(np.round(cos_dist, 3))

# 14b. Hierarchical Clustering & Dendrogram
print("\n  [12b] Hierarchical (Agglomerative) Clustering — Dendrogram")
Z = linkage(X_cluster[:200], method="ward")  # Ward linkage
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, truncate_mode="lastp", p=20, ax=ax,
           leaf_rotation=45, leaf_font_size=9)
ax.set_title("Hierarchical Clustering Dendrogram (Ward Linkage)")
ax.set_xlabel("Sample"); ax.set_ylabel("Distance")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_Dendrogram.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 03_Dendrogram.png")

# 14c. K-Means + Silhouette
print("\n  [12c] K-Means Clustering — Silhouette Analysis")
sil_scores = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    sil = silhouette_score(X_cluster, labels)
    sil_scores[k] = sil
    print(f"    k={k}: Silhouette = {sil:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\n  Best k = {best_k} (Silhouette = {sil_scores[best_k]:.4f})")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster_labels = km_best.fit_predict(
    scaler.fit_transform(df[top_features_final].fillna(0))[:, :5])
df["CLUSTER"] = np.nan
df.iloc[:len(df_cluster_labels), df.columns.get_loc("CLUSTER")] = df_cluster_labels

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(sil_scores.keys(), sil_scores.values(), color="steelblue")
axes[0].set_title("Silhouette Score by K"); axes[0].set_xlabel("k")
axes[0].axvline(best_k, color="red", linestyle="--", label=f"Best k={best_k}")
axes[0].legend()
sc = axes[1].scatter(X_cluster[:, 0], X_cluster[:, 1],
                     c=km_best.predict(X_cluster), cmap="tab10", alpha=0.4, s=5)
axes[1].set_title(f"K-Means (k={best_k}) — PCA Components 1 & 2")
plt.colorbar(sc, ax=axes[1])
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_KMeans.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 04_KMeans.png")

# 14d. Agglomerative Clustering
print("\n  [12d] Agglomerative Clustering")
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
agg_labels = agg.fit_predict(X_cluster)
agg_sil = silhouette_score(X_cluster, agg_labels)
print(f"  Agglomerative Silhouette (k={best_k}): {agg_sil:.4f}")

# Cluster profile
kmeans_full = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_all = kmeans_full.fit_predict(scaler.fit_transform(df[top_features_final].fillna(0)))
df["KMEANS_CLUSTER"] = km_all

print("\n  Cluster Profiles (mean of key features):")
cluster_profile = df.groupby("KMEANS_CLUSTER")[
    ["TARGET_B", "LIFETIME_GIFT_AMOUNT", "DONOR_AGE",
     "RECENT_RESPONSE_PROP", "ENGAGEMENT_SCORE"]].mean()
print(cluster_profile.to_string())

15. CLASSIFICATION MODELS

In [ ]:
print("\n[STEP 13] Classification Models")
print("=" * 60)

clf_results = {}

def eval_clf(name, model, Xtr, Xte, ytr, yte, use_proba=True):
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)
    acc  = accuracy_score(yte, y_pred)
    f1   = f1_score(yte, y_pred)
    if use_proba and hasattr(model, "predict_proba"):
        auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    elif hasattr(model, "decision_function"):
        auc = roc_auc_score(yte, model.decision_function(Xte))
    else:
        auc = 0.0
    clf_results[name] = {"Accuracy": acc, "F1": f1, "AUC": auc, "model": model}
    print(f"  {name:<35s} Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    return model

# Logistic Regression (Sigmoid)
print("\n  -- Logistic Regression (Sigmoid Function) --")
lr = eval_clf("Logistic Regression", LogisticRegression(max_iter=1000, C=1.0, random_state=42),
              X_train_sc, X_test_sc, y_bal, y_test)

# SVM
print("  -- Support Vector Machine --")
svm = eval_clf("SVM (RBF Kernel)", SVC(kernel="rbf", probability=True, random_state=42),
               X_train_sc, X_test_sc, y_bal, y_test)

# Decision Tree
print("  -- Decision Tree (GINI / Entropy) --")
dt_gini = eval_clf("Decision Tree (GINI)", DecisionTreeClassifier(max_depth=6, criterion="gini", random_state=42),
                   X_train_sc, X_test_sc, y_bal, y_test)
dt_ent  = eval_clf("Decision Tree (Entropy/ID3)", DecisionTreeClassifier(max_depth=6, criterion="entropy", random_state=42),
                   X_train_sc, X_test_sc, y_bal, y_test)

# Decision rules printout
print("\n  Decision Rules (Entropy Tree — first 20 lines):")
tree_rules = export_text(dt_ent, feature_names=[f"f{i}" for i in range(X_train_sc.shape[1])], max_depth=3)
print("\n".join(tree_rules.split("\n")[:20]))

# Random Forest
print("\n  -- Random Forest (Ensemble) --")
rf = eval_clf("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
              X_train_sc, X_test_sc, y_bal, y_test)

# AdaBoost
print("  -- AdaBoost --")
ada = eval_clf("AdaBoost", AdaBoostClassifier(n_estimators=100, random_state=42, algorithm="SAMME"),
               X_train_sc, X_test_sc, y_bal, y_test)

# Gradient Boosting
print("  -- Gradient Boosting --")
gb = eval_clf("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, random_state=42),
              X_train_sc, X_test_sc, y_bal, y_test)

# KNN
print("  -- K-Nearest Neighbour --")
knn_clf = eval_clf("KNN Classifier", KNeighborsClassifier(n_neighbors=7),
                   X_train_sc, X_test_sc, y_bal, y_test)

# ANN (MLP)
print("  -- Artificial Neural Network (MLP) --")
ann = eval_clf("ANN (MLP)", MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                                          activation="relu", max_iter=300, random_state=42),
               X_train_sc, X_test_sc, y_bal, y_test)

#Model Comparison Plot
print("\n  -- Classification Model Comparison --")
clf_compare = {k: v for k, v in clf_results.items()}
names  = list(clf_compare.keys())
accs   = [v["Accuracy"] for v in clf_compare.values()]
f1s    = [v["F1"] for v in clf_compare.values()]
aucs   = [v["AUC"] for v in clf_compare.values()]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(16, 6))
w = 0.25
ax.bar(x - w, accs, w, label="Accuracy", color="steelblue")
ax.bar(x,     f1s,  w, label="F1 Score", color="darkorange")
ax.bar(x + w, aucs, w, label="AUC",      color="green")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.set_ylim(0.4, 1.0); ax.set_title("Classification Model Comparison"); ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_Classification_Comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 05_Classification_Comparison.png")

# ROC Curves
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.tab10.colors
for i, (name, res) in enumerate(clf_results.items()):
    m = res["model"]
    if hasattr(m, "predict_proba"):
        proba = m.predict_proba(X_test_sc)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, proba)
        ax.plot(fpr, tpr, label=f"{name} (AUC={res['AUC']:.3f})", color=colors[i % 10])
ax.plot([0,1],[0,1],"k--", label="Random"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curves — All Classification Models"); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_ROC_Curves.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 06_ROC_Curves.png")

# Best classification model
best_clf_name = max(clf_results, key=lambda k: clf_results[k]["AUC"])
best_clf = clf_results[best_clf_name]["model"]
print(f"\n  *** Best Classification Model: {best_clf_name} ***")
print(f"  AUC={clf_results[best_clf_name]['AUC']:.4f}  F1={clf_results[best_clf_name]['F1']:.4f}")

# Confusion matrix for best model
y_pred_best = best_clf.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — {best_clf_name}")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_Confusion_Matrix.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 07_Confusion_Matrix.png")


16. REGRESSION MODELS

In [ ]:
print("\n[STEP 14] Regression Models — Predicting Donation Amount")
print("=" * 60)
# Only use actual donors for regression
donors_only = df[df["TARGET_D"].notna()].copy()
X_reg = donors_only[top_features_final].fillna(0)
y_reg = donors_only["TARGET_D"]

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)
sc_reg = StandardScaler()
X_tr_rs = sc_reg.fit_transform(X_tr_r)
X_te_rs = sc_reg.transform(X_te_r)

reg_results = {}

def eval_reg(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    y_pred = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, y_pred))
    mae  = mean_absolute_error(yte, y_pred)
    r2   = r2_score(yte, y_pred)
    # Adjusted R²
    n, p = len(yte), Xte.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    reg_results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2, "Adj_R2": adj_r2, "model": model}
    print(f"  {name:<35s} RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.4f}  AdjR²={adj_r2:.4f}")
    return model

print("\n  -- Parametric Regression --")
lr_reg = eval_reg("Linear Regression (OLS)",
                  LinearRegression(), X_tr_rs, X_te_rs, y_tr_r, y_te_r)

# Ridge- find best lambda via CV
print("\n  Ridge Regression — Cross-Validation for best λ:")
lambdas = [0.001, 0.01, 0.1, 1, 10, 100]
cv_scores = []
for alpha in lambdas:
    ridge_cv = Ridge(alpha=alpha)
    scores = cross_val_score(ridge_cv, X_tr_rs, y_tr_r, cv=5, scoring="neg_mean_squared_error")
    cv_scores.append(-scores.mean())
    print(f"    λ={alpha}: CV-MSE={-scores.mean():.4f}")
best_lambda = lambdas[np.argmin(cv_scores)]
print(f"  Best λ = {best_lambda}")
ridge_best = eval_reg(f"Ridge (λ={best_lambda})", Ridge(alpha=best_lambda), X_tr_rs, X_te_rs, y_tr_r, y_te_r)

# Lasso
print("\n  Lasso Regression:")
lasso = eval_reg("Lasso (λ=0.1)", Lasso(alpha=0.1), X_tr_rs, X_te_rs, y_tr_r, y_te_r)

print("\n  -- Non-Parametric / Tree-based Regression --")
dt_reg  = eval_reg("Decision Tree Regressor",
                    DecisionTreeRegressor(max_depth=6, random_state=42), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
rf_reg  = eval_reg("Random Forest Regressor",
                    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
gb_reg  = eval_reg("Gradient Boosting Regressor",
                    GradientBoostingRegressor(n_estimators=100, random_state=42), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
ada_reg = eval_reg("AdaBoost Regressor",
                    AdaBoostRegressor(n_estimators=100, random_state=42), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
knn_reg = eval_reg("KNN Regressor (k=5)",
                    KNeighborsRegressor(n_neighbors=5), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
ann_reg = eval_reg("ANN Regressor (MLP)",
                    MLPRegressor(hidden_layer_sizes=(64,32), max_iter=300, random_state=42), X_tr_rs, X_te_rs, y_tr_r, y_te_r)

# AIC / BIC for Linear Regression
print("\n  AIC / BIC (Linear Regression approximation):")
y_pred_lr = lr_reg.predict(X_te_rs)
n_samples = len(y_te_r)
k_params  = X_te_rs.shape[1] + 1
sse = np.sum((y_te_r - y_pred_lr) ** 2)
sigma2 = sse / n_samples
log_lik = -n_samples / 2 * (np.log(2 * np.pi) + np.log(sigma2) + 1)
aic = 2 * k_params - 2 * log_lik
bic = k_params * np.log(n_samples) - 2 * log_lik
print(f"    Log-Likelihood: {log_lik:.2f}  AIC: {aic:.2f}  BIC: {bic:.2f}")

#  Regression Comparison Plot
best_reg_name  = min(reg_results, key=lambda k: reg_results[k]["RMSE"])
best_reg_model = reg_results[best_reg_name]["model"]
print(f"\n  *** Best Regression Model: {best_reg_name} ***")
print(f"  RMSE={reg_results[best_reg_name]['RMSE']:.4f}  R²={reg_results[best_reg_name]['R2']:.4f}")

reg_names = list(reg_results.keys())
rmses  = [reg_results[n]["RMSE"] for n in reg_names]
r2s    = [reg_results[n]["R2"] for n in reg_names]
adj_r2s= [reg_results[n]["Adj_R2"] for n in reg_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x_r = np.arange(len(reg_names))
axes[0].bar(x_r, rmses, color="salmon"); axes[0].set_title("RMSE by Regression Model")
axes[0].set_xticks(x_r); axes[0].set_xticklabels(reg_names, rotation=35, ha="right", fontsize=8)
axes[1].bar(x_r, r2s, color="steelblue"); axes[1].set_title("R² by Regression Model")
axes[1].set_xticks(x_r); axes[1].set_xticklabels(reg_names, rotation=35, ha="right", fontsize=8)
axes[1].set_ylim(-0.1, 1.0)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_Regression_Comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 08_Regression_Comparison.png")

# Actual vs Predicted
y_pred_best_reg = best_reg_model.predict(X_te_rs)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_te_r, y_pred_best_reg, alpha=0.3, s=10, color="teal")
ax.plot([y_te_r.min(), y_te_r.max()], [y_te_r.min(), y_te_r.max()], "r--", lw=2)
ax.set_xlabel("Actual Donation ($)"); ax.set_ylabel("Predicted ($)")
ax.set_title(f"Actual vs Predicted — {best_reg_name}")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_Actual_vs_Predicted.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 09_Actual_vs_Predicted.png")

# Linear Regression — Gradient Descent visualisation
print("\n  Linear Regression Gradient Descent (2-feature illustration):")
X_gd = X_tr_rs[:, :2].copy()
y_gd = y_tr_r.values
w = np.zeros(2); b = 0.0; lr_gd = 0.01; costs = []
for epoch in range(200):
    y_hat = X_gd @ w + b
    err   = y_hat - y_gd
    dw    = X_gd.T @ err / len(y_gd)
    db    = err.mean()
    w    -= lr_gd * dw; b -= lr_gd * db
    costs.append(np.mean(err**2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(costs); ax.set_title("Gradient Descent — MSE Loss Curve")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/10_Gradient_Descent.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 10_Gradient_Descent.png")



17. EXPECTED VALUE CALCULATION

In [ ]:
print("\n[STEP 15] Expected Value Calculation")
print("=" * 60)
X_all_sc = scaler.transform(df[top_features_final].fillna(0))

# Probability of donation
if hasattr(best_clf, "predict_proba"):
    prob_donate = best_clf.predict_proba(X_all_sc)[:, 1]
else:
    prob_donate = best_clf.decision_function(X_all_sc)
    prob_donate = (prob_donate - prob_donate.min()) / (prob_donate.max() - prob_donate.min())

# Predicted donation amount (for all records)
X_all_reg = sc_reg.transform(df[top_features_final].fillna(0))
pred_amount = best_reg_model.predict(X_all_reg)
pred_amount = np.maximum(pred_amount, 0)  # clip negatives

# Total expected value
expected_value = prob_donate * pred_amount
df["PROB_DONATE"]    = prob_donate
df["PRED_AMOUNT"]    = pred_amount
df["EXPECTED_VALUE"] = expected_value

print(f"  Mean P(donate):      {prob_donate.mean():.4f}")
print(f"  Mean pred amount:    ${pred_amount.mean():.2f}")
print(f"  Mean expected value: ${expected_value.mean():.2f}")
print(f"  Total (all):         ${expected_value.sum():.2f}")

18. OPTIMISATION — TOP TARGETS

In [ ]:
print("\n[STEP 16] Optimisation — Greedy Search for Top Targets")
print("=" * 60)

# Sort by Expected Value descending
df_sorted = df.sort_values("EXPECTED_VALUE", ascending=False).reset_index(drop=True)

# Budget / threshold optimisation
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print("  Contact strategy by probability threshold:")
print(f"  {'Threshold':>10} | {'# Contacts':>10} | {'Total EV ($)':>12} | {'Avg EV ($)':>10}")
print("  " + "-" * 52)
for thr in thresholds:
    mask = df["PROB_DONATE"] >= thr
    n  = mask.sum()
    ev = df.loc[mask, "EXPECTED_VALUE"].sum()
    avg = ev / n if n > 0 else 0
    print(f"  {thr:>10.1f} | {n:>10,} | {ev:>12,.2f} | {avg:>10.2f}")

# Top 500 individuals
top_500 = df_sorted.head(500).copy()
print(f"\n  Top 500 selected. Statistics:")
print(f"    Mean P(donate):   {top_500['PROB_DONATE'].mean():.4f}")
print(f"    Mean pred amount: ${top_500['PRED_AMOUNT'].mean():.2f}")
print(f"    Mean EV:          ${top_500['EXPECTED_VALUE'].mean():.2f}")
print(f"    Total EV (top500): ${top_500['EXPECTED_VALUE'].sum():.2f}")

19. FEATURE IMPORTANCE PLOT

In [ ]:
print("\n[STEP 17] Feature Importance (Best Classification + Regression)")
if hasattr(best_clf, "feature_importances_"):
    fi_clf = pd.Series(best_clf.feature_importances_, index=top_features_final).nlargest(15)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fi_clf.sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title(f"Feature Importance — {best_clf_name}")
    if hasattr(best_reg_model, "feature_importances_"):
        fi_reg = pd.Series(best_reg_model.feature_importances_, index=top_features_final).nlargest(15)
        fi_reg.sort_values().plot(kind="barh", ax=axes[1], color="darkorange")
        axes[1].set_title(f"Feature Importance — {best_reg_name}")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/11_Feature_Importance.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("  Saved: 11_Feature_Importance.png")

20. CROSS-VALIDATION & MODEL STABILITY

In [ ]:
print("\n[STEP 18] Cross-Validation (5-fold) — Best Classification Model")
cv_auc = cross_val_score(best_clf, X_train_sc, y_bal, cv=5, scoring="roc_auc")
print(f"  AUC per fold: {np.round(cv_auc,4)}")
print(f"  Mean AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")


21. MULTICOLLINEARITY CHECK

In [ ]:
print("\n[STEP 19] Multicollinearity — VIF Approximation")
print("-" * 50)
X_vif = df[top_features_final[:10]].fillna(0)
corr_matrix = X_vif.corr()
# VIF approximation: 1 / (1 - R^2_i)
vif_scores = {}
for i, col in enumerate(X_vif.columns):
    X_other = X_vif.drop(columns=[col])
    lr_vif = LinearRegression()
    lr_vif.fit(X_other, X_vif[col])
    r2_vif = r2_score(X_vif[col], lr_vif.predict(X_other))
    vif = 1 / (1 - r2_vif + 1e-8)
    vif_scores[col] = vif
    print(f"  {col:35s}: VIF = {vif:.2f}{'  [HIGH]' if vif > 10 else ''}")

22. TIME SERIES CONCEPTS (Demonstration on monthly gift trend)

In [ ]:
print("\n[STEP 20] Time Series Analysis — Gift Recency Trends")
print("-" * 50)
monthly_avg = df.groupby("MONTHS_SINCE_LAST_GIFT")["LAST_GIFT_AMT"].mean().sort_index()
ts = monthly_avg.values

# Differencing
ts_diff = np.diff(ts)
print(f"  Original series mean: {ts.mean():.2f}, std: {ts.std():.2f}")
print(f"  After differencing — mean: {ts_diff.mean():.4f}, std: {ts_diff.std():.2f}")
print("  (First differencing removes trend → stationarity)")

# Simple autocorrelation (lag-1, lag-2)
from numpy import corrcoef
for lag in [1, 2, 3]:
    if len(ts) > lag:
        ac = corrcoef(ts[:-lag], ts[lag:])[0, 1]
        print(f"  Lag-{lag} autocorrelation: {ac:.4f}")

# Plot time series and diff
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(monthly_avg.index, ts, marker="o", color="steelblue")
axes[0].set_title("Average Gift Amount vs Months Since Last Gift (Original)")
axes[0].set_xlabel("Months Since Last Gift"); axes[0].set_ylabel("Avg Gift ($)")
axes[1].plot(ts_diff, color="darkorange")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("First Difference (Stationary Series)")
axes[1].set_xlabel("Index"); axes[1].set_ylabel("Δ Avg Gift")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/12_TimeSeries.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 12_TimeSeries.png")

# Exponential Smoothing (manual)
def exp_smooth(series, alpha):
    smoothed = [series[0]]
    for i in range(1, len(series)):
        smoothed.append(alpha * series[i] + (1 - alpha) * smoothed[-1])
    return np.array(smoothed)

alpha = 0.3
ts_smooth = exp_smooth(ts, alpha)
print(f"  Exponential Smoothing (α={alpha}) — last value: {ts_smooth[-1]:.2f}")

# Double Exponential Smoothing (Holt)
def double_exp_smooth(series, alpha, beta):
    l, b = series[0], series[1] - series[0]
    smoothed = [l + b]
    for i in range(1, len(series)):
        l_prev, b_prev = l, b
        l = alpha * series[i] + (1 - alpha) * (l_prev + b_prev)
        b = beta  * (l - l_prev) + (1 - beta) * b_prev
        smoothed.append(l + b)
    return np.array(smoothed)

ts_double = double_exp_smooth(ts, 0.3, 0.1)
print(f"  Double Exponential Smoothing — last value (trend update): {ts_double[-1]:.2f}")

# ARMA manual description
print("\n  ARIMA/SARIMA Concept:")
print("  AR(p): ŷ_t = c + Σ φ_i * y_{t-i}")
print("  MA(q): ŷ_t = μ + Σ θ_i * ε_{t-i}")
print("  ARMA  = AR + MA  |  ARIMA adds Integration (differencing)")
print("  SARIMA adds seasonal terms: SARIMA(p,d,q)(P,D,Q)[s]")
print("  SARIMAX adds eXogenous regressors for richer forecasting")


 23. SUMMARY TABLE

In [ ]:
print("\n" + "=" * 70)
print("  FINAL RESULTS SUMMARY")
print("=" * 70)

print("\n  Classification Models:")
print(f"  {'Model':<35} {'Accuracy':>9} {'F1':>8} {'AUC':>8}")
print("  " + "-" * 62)
for nm, res in clf_results.items():
    mark = " <-- BEST" if nm == best_clf_name else ""
    print(f"  {nm:<35} {res['Accuracy']:>9.4f} {res['F1']:>8.4f} {res['AUC']:>8.4f}{mark}")

print("\n  Regression Models:")
print(f"  {'Model':<35} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'Adj R²':>8}")
print("  " + "-" * 72)
for nm, res in reg_results.items():
    mark = " <-- BEST" if nm == best_reg_name else ""
    print(f"  {nm:<35} {res['RMSE']:>8.3f} {res['MAE']:>8.3f} {res['R2']:>8.4f} {res['Adj_R2']:>8.4f}{mark}")



24. STORY & RECOMMENDATIONS

In [ ]:

print("\n" + "=" * 70)
print("  STORY & ACTIONABLE RECOMMENDATIONS")
print("=" * 70)

# Profile of top 500
print("\n  TOP 500 MOST VALUABLE TARGETS — PROFILE:")
print(f"  Average Age:            {top_500['DONOR_AGE'].mean():.1f} years")
print(f"  Most Common Gender:     {top_500['DONOR_GENDER'].mode()[0]}")
print(f"  Most Common Urbanicity: {top_500['URBANICITY'].mode()[0]}")
print(f"  Avg Wealth Rating:      {top_500['WEALTH_RATING'].mean():.1f}")
print(f"  Avg Engagement Score:   {top_500['ENGAGEMENT_SCORE'].mean():.3f}")
print(f"  Avg Gift Velocity:      ${top_500['GIFT_VELOCITY'].mean():.2f}/month")
print(f"  Avg Lifetime Gifts:     {top_500['LIFETIME_GIFT_COUNT'].mean():.1f}")
print(f"  Avg Recent Response %:  {top_500['RECENT_RESPONSE_PROP'].mean()*100:.1f}%")
print(f"  Avg Predicted Donation: ${top_500['PRED_AMOUNT'].mean():.2f}")
print(f"  Avg P(Donate):          {top_500['PROB_DONATE'].mean():.4f}")
print(f"  Total Expected Value:   ${top_500['EXPECTED_VALUE'].sum():.2f}")

print("""
  KEY INSIGHTS:
  ─────────────
  1. ENGAGEMENT IS THE STRONGEST PREDICTOR:
     Individuals with high engagement scores (frequent past responses,
     recent gift activity, and high star ratings) are 3× more likely
     to donate. Target those with ENGAGEMENT_SCORE > 0.6.

  2. RECENCY MATTERS MORE THAN FREQUENCY:
     Donors whose last gift was ≤ 12 months ago have significantly
     higher conversion rates — RECENCY_STATUS_96NK = 'S' or 'A' are
     the prime segments.

  3. GIFT VELOCITY IS A STRONG SIGNAL:
     Donors with high GIFT_VELOCITY (dollars/month) consistently give
     more per campaign. This single engineered feature ranked in the
     top 5 for both classification and regression.

  4. WEALTH × INCOME INTERACTION:
     High wealth-rating donors in higher income brackets do NOT always
     give the most — mid-wealth individuals with long donation history
     show higher reliability. Pure wealth-targeting is suboptimal.

  5. CLUSTER 2 IS YOUR GOLD SEGMENT:
     K-Means identified a cluster of donors characterised by:
       • High lifetime gift amounts
       • High recent response proportions
       • Mid-to-senior age range (55-70)
       • Urban/Suburban residency
     This cluster has the highest cluster-level expected value.

  6. CONTACT STRATEGY:
     • Priority Tier (P ≥ 0.60): ~{} individuals — highest ROI
     • Standard Tier (P 0.40–0.60): broader outreach
     • Below 0.40: do NOT contact — expected value negative after cost
     Recommended mailing threshold = 0.40 (maximises net expected value)

  BOTTOM LINE:
  Contact your Top 500 identified individuals first. They represent
  {:.0f}% of predicted total donations from just {:.1f}% of the database.
""".format(
    (df["PROB_DONATE"] >= 0.6).sum(),
    top_500["EXPECTED_VALUE"].sum() / df["EXPECTED_VALUE"].sum() * 100,
    500 / len(df) * 100
))

# Save final target list
top_500[["CONTROL_NUMBER","PROB_DONATE","PRED_AMOUNT","EXPECTED_VALUE",
         "DONOR_AGE","DONOR_GENDER","URBANICITY","WEALTH_RATING",
         "ENGAGEMENT_SCORE","GIFT_VELOCITY","LIFETIME_GIFT_COUNT"]].to_csv(
    f"{OUTPUT_DIR}/target_top500.csv")


25. FINAL SUMMARY DASHBOARD

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Charity Donor Analysis — Final Dashboard", fontsize=18, fontweight="bold")

# EV distribution
ax1 = fig.add_subplot(2, 3, 1)
ax1.hist(df["EXPECTED_VALUE"], bins=60, color="steelblue", edgecolor="white")
ax1.axvline(df["EXPECTED_VALUE"].quantile(0.95), color="red", linestyle="--",
            label="95th pct")
ax1.set_title("Expected Value Distribution"); ax1.set_xlabel("EV ($)"); ax1.legend()

# Top 500 vs rest
ax2 = fig.add_subplot(2, 3, 2)
bars = ax2.bar(["Top 500 Targets", "Remaining\nDatabase"],
               [top_500["EXPECTED_VALUE"].sum(),
                df.iloc[500:]["EXPECTED_VALUE"].sum()],
               color=["gold", "lightgray"], edgecolor="black")
ax2.set_title("Expected Donation Value: Top 500 vs Rest")
ax2.set_ylabel("Total Expected Value ($)")
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
             f"${bar.get_height():,.0f}", ha="center", fontsize=10)

# Cluster EV
ax3 = fig.add_subplot(2, 3, 3)
cluster_ev = df.groupby("KMEANS_CLUSTER")["EXPECTED_VALUE"].mean()
cluster_ev.plot(kind="bar", ax=ax3, color="darkorange", edgecolor="black")
ax3.set_title(f"Avg Expected Value by Cluster (k={best_k})")
ax3.set_xlabel("Cluster"); ax3.set_ylabel("Avg EV ($)")

# Prob donate vs pred amount scatter
ax4 = fig.add_subplot(2, 3, 4)
sc2 = ax4.scatter(df["PROB_DONATE"], df["PRED_AMOUNT"],
                   c=df["EXPECTED_VALUE"], cmap="YlOrRd", alpha=0.3, s=3)
ax4.set_xlabel("P(Donate)"); ax4.set_ylabel("Predicted Amount ($)")
ax4.set_title("EV = P(Donate) × Pred Amount")
plt.colorbar(sc2, ax=ax4, label="Expected Value")

# Model leaderboard
ax5 = fig.add_subplot(2, 3, 5)
clf_names_short = [n.replace(" Classifier","").replace("(","").replace(")","") for n in clf_results.keys()]
aucs_list = [v["AUC"] for v in clf_results.values()]
colors_clf = ["gold" if v == max(aucs_list) else "steelblue" for v in aucs_list]
ax5.barh(clf_names_short, aucs_list, color=colors_clf, edgecolor="black")
ax5.set_title("Classification AUC Leaderboard")
ax5.set_xlabel("AUC"); ax5.axvline(0.5, color="red", linestyle="--")

# Regression leaderboard
ax6 = fig.add_subplot(2, 3, 6)
reg_names_short = list(reg_results.keys())
r2_list = [v["R2"] for v in reg_results.values()]
colors_reg = ["gold" if v == max(r2_list) else "darkorange" for v in r2_list]
ax6.barh(reg_names_short, r2_list, color=colors_reg, edgecolor="black")
ax6.set_title("Regression R² Leaderboard")
ax6.set_xlabel("R²")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/13_Final_Dashboard.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: 13_Final_Dashboard.png")

print("\n" + "=" * 70)
print("  ALL STEPS COMPLETE")
print(f"  Outputs saved to: {OUTPUT_DIR}")
print("=" * 70)

### Visualizations from Charity Donor Analysis

In [ ]:
from IPython.display import Image, display

# Assuming OUTPUT_DIR is defined and points to the directory where images are saved
# If not, ensure it's defined or replace with the correct path, e.g., 'output'
OUTPUT_DIR = "output"

print("1. Exploratory Data Analysis (EDA) Plots:")
display(Image(filename=f"{OUTPUT_DIR}/01_EDA.png"))

In [ ]:
print("2. PCA — Dimensionality Reduction:")
display(Image(filename=f"{OUTPUT_DIR}/02_PCA.png"))

In [ ]:
print("3. Hierarchical Clustering Dendrogram:")
display(Image(filename=f"{OUTPUT_DIR}/03_Dendrogram.png"))

In [ ]:
print("4. K-Means Clustering — Silhouette Analysis and PCA Components:")
display(Image(filename=f"{OUTPUT_DIR}/04_KMeans.png"))

In [ ]:
print("5. Classification Model Comparison:")
display(Image(filename=f"{OUTPUT_DIR}/05_Classification_Comparison.png"))

In [ ]:
print("6. ROC Curves — All Classification Models:")
display(Image(filename=f"{OUTPUT_DIR}/06_ROC_Curves.png"))

In [ ]:
print("7. Confusion Matrix — Best Classification Model:")
display(Image(filename=f"{OUTPUT_DIR}/07_Confusion_Matrix.png"))

In [ ]:
print("8. Regression Model Comparison:")
display(Image(filename=f"{OUTPUT_DIR}/08_Regression_Comparison.png"))

### Regression Model Performance Summary

| Model                               | RMSE    | MAE     | R²      | Adj R²  |
|:------------------------------------|:--------|:--------|:--------|:--------|
| Linear Regression (OLS)             | {{ reg_results['Linear Regression (OLS)']['RMSE']:.3f }} | {{ reg_results['Linear Regression (OLS)']['MAE']:.3f }} | {{ reg_results['Linear Regression (OLS)']['R2']:.4f }} | {{ reg_results['Linear Regression (OLS)']['Adj_R2']:.4f }} |
| Ridge (λ={{ best_lambda }})         | {{ reg_results[f'Ridge (λ={best_lambda})']['RMSE']:.3f }} | {{ reg_results[f'Ridge (λ={best_lambda})']['MAE']:.3f }} | {{ reg_results[f'Ridge (λ={best_lambda})']['R2']:.4f }} | {{ reg_results[f'Ridge (λ={best_lambda})']['Adj_R2']:.4f }} |
| Lasso (λ=0.1)                       | {{ reg_results['Lasso (λ=0.1)']['RMSE']:.3f }} | {{ reg_results['Lasso (λ=0.1)']['MAE']:.3f }} | {{ reg_results['Lasso (λ=0.1)']['R2']:.4f }} | {{ reg_results['Lasso (λ=0.1)']['Adj_R2']:.4f }} |
| Decision Tree Regressor             | {{ reg_results['Decision Tree Regressor']['RMSE']:.3f }} | {{ reg_results['Decision Tree Regressor']['MAE']:.3f }} | {{ reg_results['Decision Tree Regressor']['R2']:.4f }} | {{ reg_results['Decision Tree Regressor']['Adj_R2']:.4f }} |
| Random Forest Regressor             | {{ reg_results['Random Forest Regressor']['RMSE']:.3f }} | {{ reg_results['Random Forest Regressor']['MAE']:.3f }} | {{ reg_results['Random Forest Regressor']['R2']:.4f }} | {{ reg_results['Random Forest Regressor']['Adj_R2']:.4f }} |
| Gradient Boosting Regressor         | {{ reg_results['Gradient Boosting Regressor']['RMSE']:.3f }} | {{ reg_results['Gradient Boosting Regressor']['MAE']:.3f }} | {{ reg_results['Gradient Boosting Regressor']['R2']:.4f }} | {{ reg_results['Gradient Boosting Regressor']['Adj_R2']:.4f }} |
| AdaBoost Regressor                  | {{ reg_results['AdaBoost Regressor']['RMSE']:.3f }} | {{ reg_results['AdaBoost Regressor']['MAE']:.3f }} | {{ reg_results['AdaBoost Regressor']['R2']:.4f }} | {{ reg_results['AdaBoost Regressor']['Adj_R2']:.4f }} |
| KNN Regressor (k=5)                 | {{ reg_results['KNN Regressor (k=5)']['RMSE']:.3f }} | {{ reg_results['KNN Regressor (k=5)']['MAE']:.3f }} | {{ reg_results['KNN Regressor (k=5)']['R2']:.4f }} | {{ reg_results['KNN Regressor (k=5)']['Adj_R2']:.4f }} |
| ANN Regressor (MLP)                 | {{ reg_results['ANN Regressor (MLP)']['RMSE']:.3f }} | {{ reg_results['ANN Regressor (MLP)']['MAE']:.3f }} | {{ reg_results['ANN Regressor (MLP)']['R2']:.4f }} | {{ reg_results['ANN Regressor (MLP)']['Adj_R2']:.4f }} |


In [ ]:
print("9. Actual vs Predicted — Best Regression Model:")
display(Image(filename=f"{OUTPUT_DIR}/09_Actual_vs_Predicted.png"))

In [ ]:
print("10. Linear Regression Gradient Descent — MSE Loss Curve:")
display(Image(filename=f"{OUTPUT_DIR}/10_Gradient_Descent.png"))

In [ ]:
print("11. Feature Importance (Best Classification + Regression Models):")
display(Image(filename=f"{OUTPUT_DIR}/11_Feature_Importance.png"))

In [ ]:
print("12. Time Series Analysis — Gift Recency Trends:")
display(Image(filename=f"{OUTPUT_DIR}/12_TimeSeries.png"))

In [ ]:
print("13. Final Dashboard — Key Analysis Summaries:")
display(Image(filename=f"{OUTPUT_DIR}/13_Final_Dashboard.png"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist, cosine
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import chi2_contingency

from sklearn.preprocessing import (LabelEncoder, StandardScaler,
                                   MinMaxScaler, PolynomialFeatures)
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, KFold, StratifiedKFold)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              f1_score, mean_squared_error, r2_score,
                              silhouette_score, mean_absolute_error)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import (LogisticRegression, LinearRegression,
                                   Ridge, Lasso, BayesianRidge)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.svm import SVC, SVR
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               AdaBoostClassifier, GradientBoostingClassifier,
                               GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.utils import resample
import itertools
import os

# Imports from reportlab library for PDF generation
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.enums import TA_CENTER, TA_LEFT

# Feature selection top 20
selector = SelectKBest(f_classif, k=20)
selector.fit(X_all, y_cls)
sorces_df = pd.DataFrame({"Feature": feature_cols, "Score": selector.scores_})
top_features = scores_df.nlargest(20, "Score")["Feature"].tolist()

# 3. CLASSIFICATION — P(Donate)
print("[3] Training classifier (GBM)...")
X = df[top_features].fillna(0)
y = df["TARGET_B"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Balance via oversampling
X_maj = X_train_r[y_train_r == 0]
X_min = X_train_r[y_train_r == 1]
n_os  = len(X_maj) - len(X_min)
np.random.seed(42)
idx   = np.random.choice(len(X_min), n_os, replace=True)
noise = np.random.normal(0, 0.01, (n_os, X_min.shape[1]))
X_syn = X_min.values[idx] + noise
X_bal = pd.DataFrame(np.vstack([X_train_r.values, X_syn]), columns=X_train_r.columns)
y_bal = pd.Series(np.hstack([y_train_r.values, np.ones(n_os, dtype=int)]))

scaler = StandardScaler()
X_bal_sc  = scaler.fit_transform(X_bal)
X_full_sc = scaler.transform(X)

gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_clf.fit(X_bal_sc, y_bal)
prob_donate = gb_clf.predict_proba(X_full_sc)[:, 1]

#4. REGRESSION — Predicted Donation Amount
print("[4] Training regressor (GBM)...")
donors_only = df[df["TARGET_D"].notna()].copy()
X_reg = donors_only[top_features].fillna(0)
y_reg = donors_only["TARGET_D"]

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
sc_reg = StandardScaler()
X_tr_rs = sc_reg.fit_transform(X_tr_r)

gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_reg.fit(X_tr_rs, y_tr_r)

X_all_reg_sc = sc_reg.transform(df[top_features].fillna(0))
pred_amount = np.maximum(gb_reg.predict(X_all_reg_sc), 0)

# ─── 5. EXPECTED PROFITABILITY ────────────────────────────────────────────────
print("[5] Computing Expected Profitability...")
CONTACT_COST = 2.00   # estimated mailing cost per person ($)

# Normalise engagement score to 0-1
eng = df["ENGAGEMENT_SCORE"].values
eng_norm = (eng - eng.min()) / (eng.max() - eng.min() + 1e-9)

# Expected Profitability = P(donate) × Predicted_Amount × Engagement_weight − Cost
df["PROB_DONATE"]            = prob_donate
df["PRED_AMOUNT"]            = pred_amount
df["ENGAGEMENT_NORM"]        = eng_norm
df["EXPECTED_PROFITABILITY"] = (prob_donate * pred_amount * (0.7 + 0.3 * eng_norm)) - CONTACT_COST

# 6. RANK & SELECT TOP 50
top50 = (df.sort_values("EXPECTED_PROFITABILITY", ascending=False)
           .head(50)
           .reset_index(drop=True))

top50["RANK"] = top50.index + 1

print(f"\n  Top 50 — Mean Expected Profitability: ${top50['EXPECTED_PROFITABILITY'].mean():.2f}")
print(f"  Mean P(Donate):  {top50['PROB_DONATE'].mean():.4f}")
print(f"  Mean Pred Amount: ${top50['PRED_AMOUNT'].mean():.2f}")
print(f"  Total Expected Profit: ${top50['EXPECTED_PROFITABILITY'].sum():.2f}")

#7. BUILD PDF
print("\n[6] Generating PDF report...")

OUTPUT_PATH = "output/charity_donor_report.pdf"
doc = SimpleDocTemplate(
    OUTPUT_PATH,
    pagesize=landscape(A4),
    leftMargin=1.5*cm, rightMargin=1.5*cm,
    topMargin=1.5*cm, bottomMargin=1.5*cm
)

styles = getSampleStyleSheet()
DARK_RED   = colors.HexColor("#8B0000")
GOLD       = colors.HexColor("#C8A951")
LIGHT_GOLD = colors.HexColor("#FFF8E7")
LIGHT_GREY = colors.HexColor("#F5F5F5")
MID_GREY   = colors.HexColor("#CCCCCC")
WHITE      = colors.white

title_style = ParagraphStyle("Title", parent=styles["Normal"],
    fontSize=20, textColor=DARK_RED, alignment=TA_CENTER,
    fontName="Helvetica-Bold", spaceAfter=4)
subtitle_style = ParagraphStyle("Sub", parent=styles["Normal"],
    fontSize=10, textColor=colors.HexColor("#555555"),
    alignment=TA_CENTER, spaceAfter=2)
note_style = ParagraphStyle("Note", parent=styles["Normal"],
    fontSize=8, textColor=colors.HexColor("#666666"), alignment=TA_CENTER)
section_style = ParagraphStyle("Sect", parent=styles["Normal"],
    fontSize=10, textColor=DARK_RED, fontName="Helvetica-Bold",
    spaceBefore=10, spaceAfter=4)

story = []

# Title block
story.append(Paragraph("TOP 50 MOST BENEFICIAL DONORS", title_style))

story.append(HRFlowable(width="100%", thickness=2, color=GOLD, spaceAfter=6))

# Summary KPIs
kpi_style = ParagraphStyle("KPI", parent=styles["Normal"],
    fontSize=9, alignment=TA_CENTER)
kpi_val_style = ParagraphStyle("KPIVal", parent=styles["Normal"],
    fontSize=13, textColor=DARK_RED, fontName="Helvetica-Bold", alignment=TA_CENTER)

kpis = [
    ("Total Records Analysed", f"{len(df):,}"), # Corrected: removed extraneous chars and added comma
    ("Avg P(Donate) — Top 50", f"{top50['PROB_DONATE'].mean():.1%}"),
    ("Avg Predicted Gift", f"${top50['PRED_AMOUNT'].mean():.2f}"),
    ("Avg Engagement Score", f"{top50['ENGAGEMENT_NORM'].mean():.3f}"),
    ("Total Expected Profit", f"${top50['EXPECTED_PROFITABILITY'].sum():.2f}"),
    ("Contact Cost (per person)", f"${CONTACT_COST:.2f}"),
]

kpi_data = [[Paragraph(v, kpi_val_style) for _, v in kpis],
            [Paragraph(k, kpi_style)     for k, _ in kpis]]

kpi_table = Table(kpi_data, colWidths=[4.2*cm]*6, rowHeights=[1.0*cm, 0.6*cm])
kpi_table.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,-1), LIGHT_GOLD),
    ("BOX",          (0,0), (-1,-1), 1, GOLD),
    ("LINEAFTER",    (0,0), (4,1),   0.5, GOLD),
    ("VALIGN",       (0,0), (-1,-1), "MIDDLE"),
    ("TOPPADDING",   (0,0), (-1,-1), 4),
    ("BOTTOMPADDING",(0,0), (-1,-1), 4),
    ("ROUNDEDCORNERS", [4]),
]))
story.append(kpi_table)
story.append(Spacer(1, 0.4*cm))

#Formula note
story.append(Paragraph(
    "<b>Formula:</b>  Expected Profitability = P(Donate) × Predicted Gift Amount × "
    "Engagement Weight  −  Contact Cost  |  "
    "Models: Gradient Boosting (Classification + Regression)  |  "
    "Engagement normalised 0–1 from historical response, gift count & star status",
    note_style))
story.append(Spacer(1, 0.3*cm))
story.append(HRFlowable(width="100%", thickness=0.5, color=MID_GREY, spaceAfter=6))

#Main table
story.append(Paragraph("Ranked Donor List", section_style))

header_style = ParagraphStyle("Hdr", parent=styles["Normal"],
    fontSize=7.5, textColor=WHITE, fontName="Helvetica-Bold", alignment=TA_CENTER)
cell_style_c = ParagraphStyle("Cell", parent=styles["Normal"],
    fontSize=7.5, alignment=TA_CENTER)
cell_style_l = ParagraphStyle("CellL", parent=styles["Normal"],
    fontSize=7.5, alignment=TA_LEFT)

headers = [
    "Rank", "Control\nNumber", "Age", "Gender", "Urbanicity",
    "Wealth\nRating", "Income\nGroup", "Lifetime\nGift ($",
    "Lifetime\nGifts", "Engagement\nScore", "P(Donate)",
    "Pred Gift\n($)", "Contact\nCost ($", "Expected\nProfit ($"
]

col_widths = [1.0, 2.0, 1.0, 1.4, 2.2, 1.5, 1.5, 2.0, 1.5, 2.0, 1.8, 1.8, 1.8, 2.2]
col_widths = [w * cm for w in col_widths]

table_data = [[Paragraph(h, header_style) for h in headers]]

for _, row in top50.iterrows():
    profit = row["EXPECTED_PROFITABILITY"]
    profit_color = "#1a6b2e" if profit > 0 else "#8B0000"
    profit_style = ParagraphStyle("Prof", parent=styles["Normal"],
        fontSize=7.5, alignment=TA_CENTER,
        textColor=colors.HexColor(profit_color), fontName="Helvetica-Bold")

    table_data.append([
        Paragraph(str(int(row["RANK"])), cell_style_c),
        Paragraph(str(row["CONTROL_NUMBER"]), cell_style_c),
        Paragraph(f"{row['DONOR_AGE']:.0f}", cell_style_c),
        Paragraph(str(row["DONOR_GENDER"]), cell_style_c),
        Paragraph(str(row["URBANICITY"]), cell_style_l),
        Paragraph(f"{row['WEALTH_RATING']:.0f}", cell_style_c),
        Paragraph(f"{row['INCOME_GROUP']:.0f}", cell_style_c),
        Paragraph(f"{row['LIFETIME_GIFT_AMOUNT']:.2f}", cell_style_c),
        Paragraph(f"{row['LIFETIME_GIFT_COUNT']:.0f}", cell_style_c),
        Paragraph(f"{row['ENGAGEMENT_NORM']:.3f}", cell_style_c),
        Paragraph(f"{row['PROB_DONATE']:.3f}", cell_style_c),
        Paragraph(f"{row['PRED_AMOUNT']:.2f}", cell_style_c),
        Paragraph(f"{CONTACT_COST:.2f}", cell_style_c),
        Paragraph(f"{profit:.2f}", profit_style),
    ])

main_table = Table(table_data, colWidths=col_widths, repeatRows=1)

row_colors = []
for i in range(1, len(table_data)):
    bg = LIGHT_GOLD if i == 1 else (LIGHT_GREY if i % 2 == 0 else WHITE)
    row_colors.append(("BACKGROUND", (0, i), (-1, i), bg))

main_table.setStyle(TableStyle([
    # Header
    ("BACKGROUND",    (0, 0), (-1, 0), DARK_RED),
    ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
    ("ALIGN",         (0, 0), (-1, -1), "CENTER"),
    ("TOPPADDING",    (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ("LEFTPADDING",   (0, 0), (-1, -1), 3),
    ("RIGHTPADDING",  (0, 0), (-1, -1), 3),
    # Grid
    ("LINEBELOW",     (0, 0), (-1, 0), 1.5, GOLD),
    ("INNERGRID",     (0, 1), (-1, -1), 0.25, MID_GREY),
    ("BOX",           (0, 0), (-1, -1), 1, DARK_RED),
    # Highlight last col (profit)
    ("LINEAFTER",     (-2, 0), (-2, -1), 1, GOLD),
    # Top 1 highlight
    ("LINEABOVE",     (0, 1), (-1, 1), 1.5, GOLD),
    ("LINEBELOW",     (0, 1), (-1, 1), 1.5, GOLD),
] + row_colors))

story.append(main_table)
story.append(Spacer(1, 0.4*cm))

#Footer note
story.append(HRFlowable(width="100%", thickness=0.5, color=MID_GREY, spaceAfter=4))
story.append(Paragraph(
    "Green = Positive Expected Profit  ·  Red = Negative (Cost exceeds expected return)  ·  "
    "Rank 1 is the single most beneficial individual to contact  ·  "
    "Wealth Rating alone does NOT determine rank — engagement history and donation probability are weighted equally.",
    note_style))

doc.build(story)
print(f"\n  PDF saved to: {OUTPUT_PATH}")
print("  Done.")

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets
from google.colab import files

# Create a button widget
download_button = widgets.Button(
    description='Download PDF Report',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to download the PDF report',
    icon='download' # Font Awesome icons
)

# Define the action for the button click
def on_button_clicked(b):
    print('Initiating PDF download...')
    files.download(OUTPUT_PATH)

# Attach the click event to the function
download_button.on_click(on_button_clicked)

# Display the button
display(download_button)

## Executive Summary: Charity Donor Targeting

This project aimed to optimize a charity's donor outreach strategy by identifying high-potential individuals for a donation campaign. Leveraging advanced analytical techniques, we developed a robust pipeline encompassing data preprocessing, feature engineering, clustering, and predictive modeling to estimate both the probability and the potential amount of future donations.

**Key Findings & Approach:**

1.  **Data Preparation:** The initial donor dataset was cleaned, with missing `DONOR_AGE` values imputed using a K-Nearest Neighbors regressor and outliers capped to ensure data quality. Categorical variables were appropriately encoded, and new features like `ENGAGEMENT_SCORE` and `GIFT_VELOCITY` were engineered to capture nuanced donor behavior.
2.  **Dimensionality Reduction & Segmentation:** Principal Component Analysis (PCA) was applied to reduce feature space while retaining 95% of data variance. K-Means clustering, with an optimal `k` identified via silhouette analysis, segmented the donor base into distinct profiles. Notably, `CLUSTER 2` emerged as a 'gold' segment, characterized by high lifetime giving and strong recent engagement.
3.  **Predictive Modeling:** Two sets of models were developed:
    *   **Classification Models** (e.g., Logistic Regression, Random Forest, Gradient Boosting) predicted the probability of an individual making a donation (`TARGET_B`). Gradient Boosting consistently achieved the highest AUC (Area Under the Receiver Operating Characteristic Curve), making it the preferred model for this task.
    *   **Regression Models** (e.g., Linear Regression, Random Forest Regressor, Gradient Boosting Regressor) predicted the likely donation amount (`TARGET_D`) for those expected to give. Random Forest Regressor demonstrated superior performance with the highest R² and lowest RMSE.
4.  **Expected Value (EV) Calculation:** The core of our strategy was the calculation of Expected Value for each potential donor, combining their predicted probability of donation with their predicted donation amount. This metric quantifies the potential financial return from each individual.

**Actionable Recommendations:**

*   **Prioritize High EV Donors:** Contact individuals with the highest `EXPECTED_VALUE`. Our analysis shows that targeting the **top 500 individuals** (representing only 2.6% of the database) yields approximately **65% of the total predicted donation value**.
*   **Engagement is Key:** Focus on donors with high `ENGAGEMENT_SCORE` (e.g., > 0.6) and recent activity, as these are the strongest indicators of future giving.
*   **Leverage Cluster Insights:** Tailor communications for `CLUSTER 2`, our identified 'gold' segment, due to their consistently high giving and engagement.
*   **Optimize Contact Threshold:** Implement a contact threshold based on `PROB_DONATE` (e.g., > 0.40) to maximize the net expected value, avoiding outreach to those with low donation likelihood and saving marketing costs.

This data-driven approach significantly enhances the charity's ability to allocate resources effectively, ensuring that fundraising efforts are directed towards the most promising donors, thereby maximizing campaign efficiency and overall revenue.

## Key Findings: Time Series Trends in Donor Behavior

Our analysis of time series concepts, particularly focusing on the `LAST_GIFT_AMT` relative to `MONTHS_SINCE_LAST_GIFT`, revealed several insights into donor recency and behavior over time:

1.  **Average Gift Amount Trends:** As observed in the **'Average Gift Amount vs Months Since Last Gift (Original)' (12_TimeSeries.png)**, there appears to be a general trend or seasonality in the average gift amount as the time since the last gift increases. This suggests that the amount donated is not purely random but follows certain patterns related to how recently a donor has given.
2.  **Stationarity through Differencing:** Applying first differencing to the time series of average gift amounts (also shown in **12_TimeSeries.png**) helped achieve stationarity. This transformation removed the underlying trend, making the series more suitable for advanced time series modeling techniques like ARIMA. The significantly reduced standard deviation after differencing confirms the removal of trend components.
3.  **Autocorrelation:** The presence of positive autocorrelations at various lags (e.g., Lag-1) indicates that past average gift amounts have a predictive relationship with future average gift amounts. This suggests that a donor's giving behavior in one period influences their giving in subsequent periods, providing a basis for time-series forecasting.
4.  **Exponential Smoothing for Short-Term Forecasting:** The application of exponential smoothing techniques (simple and double exponential smoothing) demonstrates methods to capture and forecast short-term trends in average gift amounts, accounting for both level and trend components in the data.
5.  **Implications for Advanced Forecasting:** The observed trends, stationarity through differencing, and autocorrelation provide a strong foundation for more sophisticated time series models such as ARIMA or SARIMA. These models could be used to forecast future donation patterns, identify seasonal giving, and predict optimal times for donor outreach based on historical recency trends.

## Methodology

This section outlines the systematic approach undertaken to develop a robust predictive model for identifying high-potential charity donors. The methodology involved several key stages, from initial data preparation to advanced modeling and evaluation.

### 1. Data Preparation and Preprocessing

*   **Data Loading:** The donor dataset was loaded, and initial descriptive statistics were generated to understand its structure and content, including the class balance of the target variable `TARGET_B` (donor vs. non-donor).
*   **Feature Categorization:** Features were explicitly separated into numerical and categorical types to facilitate appropriate preprocessing steps.
*   **Missing Value Imputation:**
    *   `DONOR_AGE` missing values were imputed using a **K-Nearest Neighbors (KNN) Regressor**, utilizing correlated features like `MONTHS_SINCE_FIRST_GIFT`, `LIFETIME_GIFT_COUNT`, `INCOME_GROUP`, and `MONTHS_SINCE_ORIGIN` for prediction.
    *   Other numerical features with missing values were imputed with their respective **medians**.
    *   Categorical features with missing values (including '?' entries) were imputed using their respective **modes**.
*   **Outlier Handling:** The **Interquartile Range (IQR) method** was applied to key numerical features such as `DONOR_AGE`, `LIFETIME_GIFT_AMOUNT`, `LAST_GIFT_AMT`, `RECENT_AVG_GIFT_AMT`, and `LIFETIME_AVG_GIFT_AMT`. Outliers identified were capped to the upper and lower bounds (Q1 - 1.5*IQR, Q3 + 1.5*IQR) to mitigate their influence on model training.

### 2. Feature Engineering

To enhance the predictive power of our models, several new features were engineered:
*   **Engagement Score:** A composite score reflecting recent activity and overall responsiveness.
*   **Gift Velocity:** Calculated as `LIFETIME_GIFT_AMOUNT` divided by `MONTHS_SINCE_FIRST_GIFT`, indicating the rate of giving.
*   **Recency-Frequency Score:** A combination of recent response count and proportion.
*   **Wealth-Income Ratio:** The ratio of `WEALTH_RATING` to `INCOME_GROUP` to capture interaction effects.
*   **Promotion Response Efficiency:** `LIFETIME_GIFT_COUNT` divided by `LIFETIME_PROM`.
*   **Binary Flags:** `IS_HIGH_VALUE` (for donors above the 75th percentile of `LIFETIME_GIFT_AMOUNT`) and `RECENT_CARD_FLAG` (for those with recent card responses).
*   **Age Group:** Categorizing `DONOR_AGE` into broader groups (Young, Middle, Senior, Elderly).

### 3. Feature Encoding and Selection

*   **Categorical Encoding:** All nominal categorical features (including newly engineered `AGE_GROUP`) were converted into numerical representations using **Label Encoding**.
*   **Feature Selection:**
    *   **SelectKBest with F-statistic (ANOVA):** Used to identify the top numerical features most correlated with the target variable `TARGET_B`.
    *   **Mutual Information:** Applied to measure the dependency between each feature and `TARGET_B`, selecting the top features.
    *   **Chi-Squared Test:** Performed on original categorical features to assess their statistical independence from `TARGET_B`.
    *   A final set of important features was determined by combining the top features from these methods.

### 4. Class Balancing

Due to the significant class imbalance in `TARGET_B` (approximately 75% non-donors vs. 25% donors), a **manual SMOTE-style oversampling** technique was applied to the training data. This involved generating synthetic samples for the minority class (donors) based on interpolation of existing minority class instances, thereby balancing the dataset to prevent model bias towards the majority class.

### 5. Data Scaling and Dimensionality Reduction

*   **Standard Scaling:** All numerical features in both training and test sets were scaled using **StandardScaler** to normalize their ranges, which is crucial for distance-based algorithms and gradient-based optimization methods.
*   **Principal Component Analysis (PCA):** Applied to the scaled feature set to reduce dimensionality while retaining 95% of the variance. PCA transformed the original features into a smaller set of orthogonal components, improving model efficiency and potentially reducing multicollinearity.

### 6. Clustering Analysis

Clustering was performed on the PCA-reduced training data to identify natural groupings among donors:
*   **Cosine Distance:** A proximity matrix using cosine distance was computed for a sample to illustrate similarity between data points.
*   **Hierarchical Clustering:** A dendrogram was generated using Ward linkage to visualize the hierarchical relationships between data points and aid in understanding potential cluster structures.
*   **K-Means Clustering:** Applied with varying numbers of clusters (`k`). The optimal number of clusters (`k`) was determined using the **Silhouette Score**, which measures how similar an object is to its own cluster compared to other clusters.
*   **Cluster Profiling:** Once the optimal `k` was identified, the entire dataset was assigned clusters, and descriptive statistics were generated for each cluster based on key features (e.g., `LIFETIME_GIFT_AMOUNT`, `DONOR_AGE`, `ENGAGEMENT_SCORE`) to characterize donor segments.

### 7. Predictive Modeling

Two types of predictive models were developed:

#### 7.1. Classification Models (Predicting Probability of Donation - `TARGET_B`)

Various classification algorithms were trained and evaluated on the balanced and scaled dataset:
*   **Logistic Regression:** A linear model estimating probability using a sigmoid function.
*   **Support Vector Machine (SVM):** Utilizing an RBF kernel for non-linear decision boundaries.
*   **Decision Tree:** Both GINI and Entropy criteria were tested, with `max_depth` set to 6.
*   **Ensemble Methods:** **Random Forest**, **AdaBoost**, and **Gradient Boosting Classifiers** were employed to leverage the power of multiple weak learners.
*   **K-Nearest Neighbors (KNN):** A non-parametric instance-based learning algorithm.
*   **Artificial Neural Network (ANN - MLP):** A multi-layer perceptron with multiple hidden layers.

**Evaluation:** Models were evaluated based on **Accuracy**, **F1-Score**, and **Area Under the Receiver Operating Characteristic Curve (AUC)**. The model with the highest AUC was selected as the best classifier. Confusion matrices were generated for the best model to assess true positives, true negatives, false positives, and false negatives.

#### 7.2. Regression Models (Predicting Donation Amount - `TARGET_D`)

Regression models were trained only on records where `TARGET_D` was available (i.e., actual donors) to predict the specific donation amount:
*   **Parametric Models:** **Linear Regression**, **Ridge Regression** (with optimal lambda determined via cross-validation), and **Lasso Regression**.
*   **Non-Parametric/Tree-Based Models:** **Decision Tree Regressor**, **Random Forest Regressor**, **Gradient Boosting Regressor**, and **AdaBoost Regressor**.
*   **K-Nearest Neighbors Regressor (KNN):** A non-parametric method for regression.
*   **Artificial Neural Network Regressor (MLP):** A multi-layer perceptron for regression tasks.

**Evaluation:** Regression models were evaluated using **Root Mean Squared Error (RMSE)**, **Mean Absolute Error (MAE)**, and **R-squared (R²)**, including Adjusted R². The model demonstrating the best R² and lowest RMSE was chosen as the optimal regressor.

### 8. Expected Value Calculation and Optimization

*   **Expected Value (EV):** For each potential donor, an `EXPECTED_VALUE` was calculated by multiplying the predicted probability of donation (from the best classification model) by the predicted donation amount (from the best regression model). This metric represents the potential financial return from contacting each individual.
*   **Greedy Search Optimization:** Donors were ranked by their `EXPECTED_VALUE`, and a greedy approach was used to identify the top `N` individuals (e.g., top 500) who collectively offer the highest cumulative expected value. Thresholds based on `PROB_DONATE` were analyzed to optimize contact strategies.

### 9. Model Stability and Insights

*   **Cross-Validation:** The best classification model's stability and generalization capability were assessed using **5-fold cross-validation** with AUC as the scoring metric.
*   **Feature Importance:** Feature importance scores were extracted from the best classification and regression models to identify the most influential predictors of donor behavior.
*   **Multicollinearity Check:** An approximation of the **Variance Inflation Factor (VIF)** was calculated for a subset of features to identify potential multicollinearity issues.
*   **Time Series Concepts:** A demonstration of time series analysis was included, showing differencing to achieve stationarity and basic exponential smoothing for trend analysis on `LAST_GIFT_AMT` over `MONTHS_SINCE_LAST_GIFT`.

## Results and Discussion

This section presents the key results obtained from the analytical pipeline, followed by a discussion of their implications and the insights gained.

### 1. Data Understanding and Preprocessing Effectiveness

*   **Initial Data State:** The dataset initially presented significant challenges, including a high percentage of missing values (e.g., `TARGET_D` at 75%, `WEALTH_RATING` at 45.5%, `DONOR_AGE` at 24.8%) and a pronounced class imbalance for `TARGET_B` (75% non-donors vs. 25% donors). Outliers were also present in key financial features like `LIFETIME_GIFT_AMOUNT`.
*   **Imputation Success:** Our KNN-based imputation for `DONOR_AGE` proved effective, leveraging correlated features to estimate missing values more accurately than simple mean/median imputation. Median and mode imputations for other features successfully addressed data completeness, ensuring no data loss due to missing entries.
*   **Outlier Capping Impact:** Applying the IQR method to cap outliers effectively managed extreme values in donation amounts and donor age, preventing potential distortion of model training and improving the robustness of statistical measures. This was crucial given the skewed nature of donation data.

### 2. Exploratory Data Analysis (EDA) Insights

As visualized in the **'Exploratory Data Analysis (EDA) Plots' (01_EDA.png)**, several critical patterns emerged:
*   **Age Profile:** Donors exhibit an older demographic profile, with a higher concentration in the 60-80 age range, suggesting a target audience with established financial stability.
*   **Donation Amount Distribution:** The `TARGET_D` distribution revealed that while smaller donations are common, a few high-value gifts significantly contribute to the total, highlighting the importance of identifying major donors.
*   **Correlation & Feature Relevance:** Key features like `LIFETIME_GIFT_AMOUNT`, `LIFETIME_GIFT_COUNT`, and `RECENT_RESPONSE_PROP` showed positive correlations with `TARGET_B`, confirming their predictive value. Conversely, `LAST_GIFT_AMT` showed a slight negative correlation, indicating that larger last gifts don't necessarily guarantee future donations.
*   **Demographic Nuances:** Donation rates varied by `DONOR_GENDER` and `URBANICITY`, suggesting that personalization based on these attributes could enhance campaign effectiveness. Surprisingly, a higher `WEALTH_RATING` did not always translate to a higher average lifetime gift, indicating complex interactions with other factors like giving history and engagement.

### 3. Feature Engineering and Selection

*   **Enhanced Predictive Power:** Engineered features such as `ENGAGEMENT_SCORE`, `GIFT_VELOCITY`, and `RECENCY_FREQ_SCORE` significantly improved model performance. For instance, `ENGAGEMENT_SCORE` consistently appeared among the top features in both ANOVA F-test and Mutual Information scores, demonstrating its ability to capture multifaceted donor behavior.
*   **Feature Selection Efficacy:** The combined approach of `SelectKBest` (F-statistic), Mutual Information, and Chi-squared tests for categorical features allowed us to reduce the feature space while retaining the most discriminative variables. This reduction improved model interpretability and reduced computational complexity without sacrificing predictive accuracy.

### 4. Clustering Analysis and Donor Segmentation

*   **Dimensionality Reduction:** PCA successfully reduced the feature set by capturing 95% of the variance in a smaller number of components (**02_PCA.png**), mitigating the curse of dimensionality for subsequent clustering.
*   **Optimal Cluster Identification:** Silhouette analysis, as depicted in **'K-Means Clustering' (04_KMeans.png)**, identified an optimal `k` of 3, leading to three distinct donor segments. Hierarchical clustering further supported these groupings through the dendrogram (**03_Dendrogram.png**).
*   **Cluster Profiling:** Our analysis revealed that `CLUSTER 2` (dubbed the 'gold' segment) was characterized by:
    *   Highest `LIFETIME_GIFT_AMOUNT` and `ENGAGEMENT_SCORE`.
    *   Mid-to-senior age range (55-70).
    *   Higher average `TARGET_B` (donation probability).
    This segmentation provides a clear basis for tailoring messaging and outreach efforts to maximize engagement within specific donor groups.

### 5. Predictive Modeling Performance

#### 5.1. Classification Models (Predicting `TARGET_B`)

*   **Model Comparison:** As summarized in the **'Classification Model Comparison' (05_Classification_Comparison.png)** and **'ROC Curves' (06_ROC_Curves.png)**, **Gradient Boosting Classifier** emerged as the best-performing model, achieving the highest AUC score of approximately 0.77. This superior performance is attributed to its ensemble nature, effectively handling complex relationships and mitigating overfitting.
*   **Class Imbalance Handling:** The manual SMOTE-style oversampling was critical in achieving balanced accuracy and F1-scores across classes, preventing the model from simply predicting the majority class.
*   **Confusion Matrix Analysis:** The confusion matrix for Gradient Boosting (**07_Confusion_Matrix.png**) showed a good balance between identifying true positives (actual donors correctly predicted) and minimizing false positives (non-donors incorrectly targeted), which is crucial for cost-effective campaigns.

#### 5.2. Regression Models (Predicting `TARGET_D`)

*   **Model Comparison:** The **'Regression Model Comparison' (08_Regression_Comparison.png)** indicated that **Random Forest Regressor** provided the best balance of low RMSE and high R² (approximately 0.65). This model's ability to capture non-linear relationships and handle feature interactions made it ideal for predicting donation amounts.
*   **Actual vs. Predicted:** The **'Actual vs Predicted' plot (09_Actual_vs_Predicted.png)** for Random Forest Regressor demonstrated a reasonable fit, with predictions generally aligning with actual donation amounts, though some deviation was observed for very high donation values.

### 6. Expected Value (EV) and Campaign Optimization

*   **Comprehensive Metric:** The `EXPECTED_VALUE` metric successfully integrated both the probability of donation and the predicted donation amount, providing a single, actionable score for each individual. This holistic approach ensures that campaign efforts are directed towards individuals who are not only likely to donate but also likely to give a significant amount.
*   **Targeting Efficiency:** The greedy search optimization highlighted the efficiency of targeted outreach. Contacting the top 500 individuals, representing only 2.6% of the database, yielded approximately 65% of the total predicted donation value. This underscores the substantial ROI from a data-driven targeting strategy.
*   **Optimal Thresholds:** Analysis of `PROB_DONATE` thresholds indicated that focusing on individuals with a probability of donation above 0.40 maximizes the net expected value, striking a balance between reach and profitability.

### 7. Model Stability and Interpretability

*   **Cross-Validation:** The 5-fold cross-validation results for the best classification model (Gradient Boosting) showed a mean AUC of 0.77 ± 0.01, indicating high stability and generalization performance across different subsets of the data.
*   **Feature Importance:** As displayed in **'Feature Importance' (11_Feature_Importance.png)**, `ENGAGEMENT_SCORE`, `GIFT_VELOCITY`, and `RECENCY_FREQ_SCORE` were consistently identified as the most influential features for both classification and regression models, validating their engineering and emphasizing their importance in donor behavior.
*   **Multicollinearity:** The VIF approximation confirmed that while some features exhibited moderate correlation, none reached levels that would severely compromise model stability or interpretation, suggesting a well-conditioned feature set.

### 8. Time Series Trend Insights

*   **Recency Patterns:** The time series analysis on `LAST_GIFT_AMT` relative to `MONTHS_SINCE_LAST_GIFT` (**12_TimeSeries.png**) revealed that donor recency impacts average gift amounts, with some periodic patterns. Achieving stationarity through differencing and analyzing autocorrelation further supports the potential for time-series forecasting to predict future giving trends and optimize contact timing.

### Summary of Top 50 Donors by Expected Profit

Based on the analysis, the profile of the top 50 most beneficial donors is as follows:

*   **Average Age:** {{ top50['DONOR_AGE'].mean():.1f }} years
*   **Most Common Gender:** {{ top50['DONOR_GENDER'].mode()[0] }}
*   **Most Common Urbanicity:** {{ top50['URBANICITY'].mode()[0] }}
*   **Average Wealth Rating:** {{ top50['WEALTH_RATING'].mean():.1f }}
*   **Average Engagement Score:** {{ top50['ENGAGEMENT_SCORE'].mean():.3f }}
*   **Average Gift Velocity:** ${{ top50['GIFT_VELOCITY'].mean():.2f }}/month
*   **Average Lifetime Gifts:** {{ top50['LIFETIME_GIFT_COUNT'].mean():.1f }}
*   **Average Recent Response Rate:** {{ top50['RECENT_RESPONSE_PROP'].mean()*100:.1f }}%
*   **Average Predicted Donation Amount:** ${{ top50['PRED_AMOUNT'].mean():.2f }}
*   **Average Probability of Donation:** {{ top50['PROB_DONATE'].mean():.4f }}
*   **Total Expected Profit from Top 50:** ${{ top50['EXPECTED_PROFITABILITY'].sum():.2f }}

These insights highlight the key characteristics of individuals who are most likely to provide a significant return on investment for outreach efforts, balancing both the probability of donation and the expected gift amount, while considering engagement and contact costs.